# Réplica — Kim (2014), *Convolutional Neural Networks for Sentence Classification*

Notebook para rodar a réplica completa no Google Colab.

**Antes de começar:** `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)`.

Artigo: https://arxiv.org/abs/1408.5882

## 1. Código e dependências

In [ ]:
!git clone https://github.com/ivannerilo/replica-cnn-sentence-classification.git
%cd replica-cnn-sentence-classification
!pip install -q gensim
import torch; print('GPU:', torch.cuda.is_available())

## 2. Dados (base MR) e pré-processamento

O pré-processamento baixa os vetores word2vec do Google News (~1,6 GB) e leva alguns minutos.
Saída esperada (igual à Tabela 1 do artigo): 10662 frases, vocabulário 18765, 16448 palavras no word2vec.

In [ ]:
!python data/download_data.py
!python src/preprocess.py

## 3. Treino das 4 variantes (validação cruzada em 10 folds, 25 épocas)

Na GPU cada variante leva poucos minutos. Os logs e resultados ficam em `results/`.

In [ ]:
!python src/train.py --variant rand

In [ ]:
!python src/train.py --variant static

In [ ]:
!python src/train.py --variant nonstatic --save-embeddings

In [ ]:
!python src/train.py --variant multichannel --save-embeddings

## 4. Comparação com o artigo

In [ ]:
!python src/analysis.py

## 5. Curvas de treino/validação (fold 0)

In [ ]:
import json, matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 4, figsize=(18, 3.5), sharey=True)
for ax, v in zip(axes, ['rand', 'static', 'nonstatic', 'multichannel']):
    h = json.load(open(f'results/{v}.json'))['folds'][0]['history']
    ax.plot([e['train'] for e in h], label='treino')
    ax.plot([e['val'] for e in h], label='validação')
    ax.set_title(f'CNN-{v}'); ax.set_xlabel('época')
axes[0].set_ylabel('acurácia'); axes[0].legend()
plt.tight_layout(); plt.savefig('results/curvas.png', dpi=120); plt.show()